In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Directory and file patterns
base_dir = "/path/to/project" 

# Population directories
cha_pan = os.path.join(base_dir, "final_analysis/data/CHA/pan/bp35w60")
cha_ngs = os.path.join(base_dir, "final_analysis/data/CHA/ngs/bp35w60")
chb_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHB")
chs_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHS")

datasets = {
    "CHA_pan": cha_pan,
    "CHA_ngs": cha_ngs,
    "CHB": chb_unmasked_dir,
    "CHS": chs_unmasked_dir,
}

chroms = [str(i) for i in range(1, 23)]

# ------------------------
# File patterns
# ------------------------
file_patterns = {
    cha_pan: "CHA_recombmap_chr{chrom}_bp35w60",
    cha_ngs: "CHA_recombmap_chr{chrom}_bp35w60",
    chb_unmasked_dir: "CHB_chr{chrom}_no_mask.txt",
    chs_unmasked_dir: "CHS_chr{chrom}_no_mask.txt",
}


def load_map(directory, chrom):
    if directory not in file_patterns:
        raise ValueError(f"Unknown directory: {directory}")
    file_name = file_patterns[directory].format(chrom=chrom)
    file_path = os.path.join(directory, file_name)
    
    if directory in [cha_pan, cha_ngs]:
        df = pd.read_csv(file_path, sep="\t", header=None, names=["Start", "End", "Rec.Rate"])
    else:
        df = pd.read_csv(file_path)
    return df


chroms = [str(i) for i in range(1, 23)]



In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:

    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    # Two-pointer sweep (fast enough; map windows are usually not huge)
    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out



# remove unavailable region

pan_available = "CHM13v2.telo_cent.complement.bed"

pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region

In [ ]:
def get_rate_at_positions(df, positions):
    """
    df: Start, End, Rec.Rate
    positions: np.array of bp positions
    """
    starts = df["Start"].to_numpy()
    ends   = df["End"].to_numpy()
    rates  = df["Rec.Rate"].to_numpy()

    out = np.full(len(positions), np.nan)

    idx = np.searchsorted(starts, positions, side="right") - 1
    valid = (idx >= 0) & (positions < ends[idx])
    out[valid] = rates[idx[valid]]
    return out


In [ ]:
def mean_rate_over_windows(df, windows):
    """
    windows: list of (start, end)
    returns: np.array of mean recombination rates
    """
    out = np.full(len(windows), np.nan)

    starts = df["Start"].to_numpy()
    ends   = df["End"].to_numpy()
    rates  = df["Rec.Rate"].to_numpy()

    for i, (ws, we) in enumerate(windows):
        if we <= ws:
            continue

        idx_start = np.searchsorted(ends, ws, side="right")
        idx_end   = np.searchsorted(starts, we, side="left")

        total_len = 0.0
        total_rec = 0.0

        for j in range(idx_start, idx_end):
            ov_s = max(ws, starts[j])
            ov_e = min(we, ends[j])
            if ov_s < ov_e:
                L = ov_e - ov_s
                total_len += L
                total_rec += L * rates[j]

        if total_len > 0:
            out[i] = total_rec / total_len

    return out


In [ ]:
def sample_positions_from_allowed(allowed_df, chrom, n):
    df = allowed_df[allowed_df["chr"] == str(chrom)]
    lengths = df["End"] - df["Start"]
    probs = lengths / lengths.sum()

    intervals = df[["Start", "End"]].to_numpy()
    chosen = np.random.choice(len(intervals), size=n, p=probs)

    positions = np.array([
        np.random.randint(intervals[i][0], intervals[i][1])
        for i in chosen
    ])
    return positions


def sample_windows_from_allowed(allowed_df, chrom, n, window_size):
    """
    Sample windows that are fully contained within the union
    of allowed regions (can span multiple adjacent intervals).
    """

    df = allowed_df[allowed_df["chr"] == str(chrom)].copy()
    if df.empty:
        return []

    # Sort intervals
    df = df.sort_values(["Start", "End"])
    intervals = df[["Start", "End"]].to_numpy()

    # Chromosome bounds (allowed union span)
    chrom_start = intervals[0][0]
    chrom_end   = intervals[-1][1]

    windows = []
    attempts = 0
    max_attempts = n * 50   # prevent infinite loop

    while len(windows) < n and attempts < max_attempts:
        attempts += 1

        s = np.random.randint(chrom_start, chrom_end - window_size)
        e = s + window_size

        # Check if fully covered by allowed regions
        covered = 0

        for a_s, a_e in intervals:
            if a_e <= s:
                continue
            if a_s >= e:
                break

            ov_s = max(s, a_s)
            ov_e = min(e, a_e)

            if ov_s < ov_e:
                covered += (ov_e - ov_s)

        if covered == window_size:
            windows.append((s, e))

    return windows



In [ ]:
def collect_pairs_for_scale(scale_type, window_size=None,
                            n_per_chrom=1000, max_rate=1e-6):

    all_pairs = []

    for chrom in chroms:
        maps = {}

        for name, directory in datasets.items():
            df = load_map(directory, chrom)

            if name in ["CHA_pan", "CHA_ngs"]:
                df = clip_map_to_allowed_regions(
                    df, pan_available_region, chrom
                )

            maps[name] = df

        if scale_type == "point":
            positions = sample_positions_from_allowed(
                pan_available_region, chrom, n_per_chrom
            )

            rates = [
                get_rate_at_positions(maps[name], positions)
                for name in datasets
            ]

        else:
            windows = sample_windows_from_allowed(
                pan_available_region, chrom, n_per_chrom, window_size
            )

            rates = [
                mean_rate_over_windows(maps[name], windows)
                for name in datasets
            ]

        pairs = np.column_stack(rates)

        valid = (
            ~np.isnan(pairs).any(axis=1) &
            (pairs <= max_rate).all(axis=1)
        )

        if valid.any():
            all_pairs.append(pairs[valid])

    return np.vstack(all_pairs)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr


def plot_spearman_heatmap(combined, dataset_names, title, out_pdf):
    """
    combined: N x 4 array
    """

    n = combined.shape[1]
    corr_matrix = np.zeros((n, n))

    # Compute Spearman correlation matrix
    for i in range(n):
        for j in range(n):
            if i == j:
                corr_matrix[i, j] = 1.0
            else:
                r, _ = spearmanr(combined[:, i], combined[:, j])
                corr_matrix[i, j] = r

    fig, ax = plt.subplots(figsize=(6, 6))

    im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")

    # Add correlation values inside cells
    for i in range(n):
        for j in range(n):
            ax.text(
                j, i,
                f"{corr_matrix[i, j]:.2f}",
                ha="center", va="center",
                fontsize=12,
                color="black"
            )

    # Axis formatting
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(dataset_names, rotation=45, ha="right")
    ax.set_yticklabels(dataset_names)

    ax.set_title(title)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Spearman correlation")

    plt.tight_layout()
    fig.savefig(out_pdf)
    plt.show()
    plt.close(fig)


In [ ]:
dataset_names = ["CHA_pan", "CHA_ngs", "CHB", "CHS"]

# (1) 1 bp
print("Collecting data for 1 bp sampling...")
combined_1bp = collect_pairs_for_scale("point", n_per_chrom=1000)

plot_spearman_heatmap(
    combined_1bp,
    dataset_names,
    "Spearman correlation (1 bp sampling)",
    "corr_heatmap_1bp.pdf"
)

# (2) 1 kb
print("Collecting data for 1 kb windows...")
combined_1kb = collect_pairs_for_scale(
    "window", window_size=1_000, n_per_chrom=100
)

plot_spearman_heatmap(
    combined_1kb,
    dataset_names,
    "Spearman correlation (1 kb windows)",
    "corr_heatmap_1kb.pdf"
)

# (3) 10 kb
print("Collecting data for 10 kb windows...")
combined_10kb = collect_pairs_for_scale(
    "window", window_size=10_000, n_per_chrom=100
)

plot_spearman_heatmap(
    combined_10kb,
    dataset_names,
    "Spearman correlation (10 kb windows)",
    "corr_heatmap_10kb.pdf"
)

# (4) 100 kb
print("Collecting data for 100 kb windows...")
combined_100kb = collect_pairs_for_scale(
    "window", window_size=100_000, n_per_chrom=100
)

plot_spearman_heatmap(
    combined_100kb,
    dataset_names,
    "Spearman correlation (100 kb windows)",
    "corr_heatmap_100kb.pdf"
)
